# Scaled Dot-Product Attention

## Dot Product Attention 

Dot product attention computes the similarity between queries and keys using the dot product. The attention weights are then obtained by applying the softmax function to these dot products.

::: {.callout-tip collapse="true"}

## Dot Product Primer

In machine learning, data points are often represented as vectors in a high-dimensional space. For example, word embeddings represent words as vectors, with semantically related words often occupying nearby regions of the embedding space. To compare two vectors, we need a way to measure how similar they are.

Three commonly used measures are **Euclidean distance, cosine similarity, and dot product**. Although closely related mathematically, they capture different aspects of the relationship between vectors.

### 1. Euclidean Distance: How Far Apart Are Two Vectors?

Euclidean distance measures the straight-line distance between two points in space.

For two vectors $\mathbf{a}$ and $\mathbf{b}$:

$$
d(\mathbf{a},\mathbf{b})=\sqrt{\sum_{i=1}^{d}(a_i-b_i)^2}
$$

A smaller distance indicates that the vectors are closer together. Euclidean distance considers both the direction and magnitude of the vectors.

### 2. Cosine Similarity: How Aligned Are Two Vectors?

Cosine similarity measures the angle between two nonzero vectors, regardless of their magnitudes.

$$
\cos(\theta)=\frac{\mathbf{a}\cdot\mathbf{b}}{\|\mathbf{a}\|\|\mathbf{b}\|}
$$

- **1:** Vectors point in the same direction.
- **0:** Vectors are perpendicular.
- **−1:** Vectors point in opposite directions.

For example, the vectors $[1,2]$ and $[10,20]$ have a cosine similarity of 1, even though they are far apart in Euclidean distance.

Cosine similarity is particularly useful when the direction of a vector is more important than its magnitude.

### 3. Dot Product: Combining Direction and Magnitude

The dot product of two vectors is calculated by multiplying corresponding components and summing the results:

$$
\mathbf{a}\cdot\mathbf{b}=\sum_{i=1}^{d}a_i b_i
$$

Geometrically, the dot product can also be expressed as:

$$
\mathbf{a}\cdot\mathbf{b}=\|\mathbf{a}\|\|\mathbf{b}\|\cos(\theta)
$$

This reveals an important relationship: **the dot product combines angular alignment (cosine similarity) with vector magnitudes.**

Two vectors have a large positive dot product when they point in similar directions and have large magnitudes. Their dot product is zero when they are perpendicular and negative when they point in opposing directions.

Unlike cosine similarity, the dot product is not normalized. Consequently, two vectors can have a high dot product because of their magnitudes, even if they are not especially well aligned.

### 4. Comparing the Three Measures

Consider three vectors:

$$
\mathbf{a}=[1,2],\quad \mathbf{b}=[2,4],\quad \mathbf{c}=[-2,1]
$$

| Comparison | Euclidean Distance | Cosine Similarity | Dot Product |
| ---------- | -----------------: | ----------------: | ----------: |
| a vs. b    |               2.24 |               1.0 |          10 |
| a vs. c    |               3.16 |               0.0 |           0 |

Vectors **a** and **b** point in exactly the same direction, giving them a cosine similarity of 1, but they have different magnitudes and therefore a nonzero Euclidean distance. Vectors **a** and **c** are perpendicular, so both their cosine similarity and dot product are zero.

**Key takeaway:** Euclidean distance measures *proximity*, cosine similarity measures *directional alignment*, and dot product measures *alignment weighted by magnitude*. For unit-length vectors, dot product and cosine similarity are identical.

:::

Dot Product is preferred when the dimensionality of the queries and keys is not too large, as it is computationally efficient and works well in practice.



### Other Attention Mechanisms

Other measures of similarity between queries and keys include additive attention, cosine similarity, and learned compatibility functions. 

These mechanisms can be used when dot product attention is not suitable or when different properties are desired.

Additive attention computes the similarity between queries and keys using a feed-forward neural network. It is often used when the dimensionality of the queries and keys is large, as it can be more stable than dot product attention.

$$
\text{Additive Attention: } \text{score}(\mathbf{q}, \mathbf{k}) = \mathbf{v}^\top \tanh(\mathbf{W}_q \mathbf{q} + \mathbf{W}_k \mathbf{k})
$$

Cosine similarity measures the cosine of the angle between queries and keys, effectively normalizing their magnitudes. It is useful when the direction of the vectors is more important than their magnitude.

$$
\text{Cosine Similarity: } \text{score}(\mathbf{q}, \mathbf{k}) = \frac{\mathbf{q} \cdot \mathbf{k}}{\|\mathbf{q}\| \|\mathbf{k}\|}
$$

Learned compatibility functions use a parameterized function to compute the similarity between queries and keys. This allows the model to learn the most appropriate similarity measure for the task at hand.

$$
\text{Learned Compatibility: } \text{score}(\mathbf{q}, \mathbf{k}) = f_\theta(\mathbf{q}, \mathbf{k})
$$

## Scaled Attention 

The passage compresses three ideas: why the dot products get large, why large inputs make softmax "saturate," and why saturation kills gradients. Taking them in order:

### 1. Dot products grow with dimension

Suppose each component of a query $\mathbf{q}$ and a key $\mathbf{k}$ is roughly independent with mean 0 and variance 1. That's approximately true at initialization with standard weight init and normalized inputs. Their dot product is

$$\mathbf{q}\cdot\mathbf{k} = \sum_{i=1}^{d_k} q_i k_i$$

Each term $q_i k_i$ has mean 0 and variance 1, and you're summing $d_k$ of them. So the dot product has **variance $d_k$**, or standard deviation $\sqrt{d_k}$.

With $d_k = 64$, typical attention logits are around ±8. With $d_k = 512$, they're around ±23. Nothing has been learned yet; the size comes purely from the dimension.

### 2. Large logits make softmax nearly one-hot

Softmax depends on the *differences* between its inputs, and it's exponential in them. Compare one query's logits against three keys:

| logits | softmax output |
|---|---|
| [8, 0, −8] (unscaled, $d_k=64$) | [0.9997, 0.0003, 0.0000001] |
| [1, 0, −1] (divided by $\sqrt{64}=8$) | [0.665, 0.245, 0.090] |



In the unscaled case, the query attends to one key and ignores everything else. The model never made that decision; it's an artifact of the logit scale.

<mark>3. A one-hot softmax has near-zero gradients</mark>

This is the key step. Training works by asking: *if I nudge this input a little, how much does the output change?* That sensitivity is the gradient. If the answer is "barely at all," the model gets almost no signal about which direction to adjust.

**A nudge experiment.** Take the logits from the example above and increase the second one by 1:

| | before nudge | after nudge | change in 2nd weight |
|---|---|---|---|
| Unscaled | [8, 0, −8] → weights [0.9997, 0.0003, 0.0000] | [8, 1, −8] → weights [0.9991, 0.0009, 0.0000] | **+0.0006** |
| Scaled | [1, 0, −1] → weights [0.665, 0.245, 0.090] | [1, 1, −1] → weights [0.468, 0.468, 0.063] | **+0.22** |

The same nudge moves the scaled attention weight about 370 times more. When one logit dominates, softmax is "stuck": the winner already has nearly all the weight and the others have nearly none, so small changes in either direction hardly register.

<mark>4. Why $\sqrt{d_k}$ specifically</mark>

Dividing by $\sqrt{d_k}$ divides by the standard deviation of the dot product, which brings the variance back to 1 regardless of dimension:

$$\text{Var}\!\left(\frac{\mathbf{q}\cdot\mathbf{k}}{\sqrt{d_k}}\right) = \frac{d_k}{d_k} = 1$$

Dividing by $d_k$ would overshoot: the variance becomes $1/d_k$, the logits all sit near zero, and attention becomes nearly uniform. Not scaling at all leaves softmax saturated. The $\sqrt{d_k}$ factor is exactly what keeps logits in the range where softmax is sensitive.
